In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('..'))

from src.utils import load_full_config
from src.model import SimpleTransformer


In [ ]:
def fit_parabola(s_range, phi_values):
    X = np.stack([s_range ** 2, s_range], axis=1)
    coeffs, _, _, _ = np.linalg.lstsq(X, phi_values, rcond=None)
    A, B = coeffs
    return A, B


In [ ]:
cfg = load_full_config()

device = torch.device(
    'cuda' if torch.cuda.is_available() and cfg['system']['device'] == 'cuda' else 'cpu'
)

# Load data
data_path = os.path.join('..', cfg['paths']['mgf_data_path'])
data = torch.load(data_path, weights_only=False)

trajectories = data['trajectories']
targets = data['targets']
theta_values = data['theta_values']
s_range = data['s_range']

# Load model
model = SimpleTransformer(**cfg['architecture'])
model_path = os.path.join(
    '..',
    cfg['paths']['save_dir'],
    cfg['paths'].get('mgf_model_name', 'model_mgf.pth')
)

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    print(f"Loaded model from {model_path}")
else:
    raise FileNotFoundError(
        f"Model not found at {model_path}. Train first with python ../scripts/train.py"
    )


In [ ]:
idx = np.random.randint(0, len(trajectories))
traj = trajectories[idx:idx+1].to(device)
true_theta = theta_values[idx].item()
X_L = traj[0, -1, 0].item()

with torch.no_grad():
    preds, _ = model(traj)
    phi_pred = preds[0, -1, :].cpu().numpy()

A, B = fit_parabola(s_range, phi_pred)

pred_mean = B
pred_var = 2 * A

D = cfg['physics']['D']
dt = cfg['physics']['dt']
mu = cfg['physics']['mu']

exp_theta_dt = np.exp(-true_theta * dt)
theory_mean = exp_theta_dt * X_L + mu * (1 - exp_theta_dt)
theory_var = (D / true_theta) * (1 - np.exp(-2 * true_theta * dt))

print(f"Trajectory {idx}: theta={true_theta:.3f}, X_L={X_L:.3f}")
print(f"Predicted mean: {pred_mean:.6f} | Theoretical mean: {theory_mean:.6f}")
print(f"Predicted var:  {pred_var:.6f} | Theoretical var:  {theory_var:.6f}")


In [ ]:
traj_np = traj[0].cpu().squeeze().numpy()
timestamps = np.arange(len(traj_np)) * cfg['physics']['dt']

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(timestamps, traj_np, 'b-', alpha=0.7, label='Trajectory')
ax.scatter([timestamps[-1]], [traj_np[-1]], color='black', s=60, label='X_L')
ax.axhline(y=theory_mean, color='green', linestyle='--', linewidth=2, label='Theoretical mean')
ax.axhline(y=pred_mean, color='red', linestyle='--', linewidth=2, label='Predicted mean')
ax.fill_between(
    timestamps,
    pred_mean - np.sqrt(pred_var),
    pred_mean + np.sqrt(pred_var),
    color='red',
    alpha=0.2,
    label='Predicted ±1σ'
)
ax.set_xlabel('Time')
ax.set_ylabel('X(t)')
ax.set_title('MGF-Based Mean/Variance Prediction')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
